In [ ]:
import os
from amendements_intelligents.summary.llm_clients import (
    EtalabAPIClient,
    LLMAPIClient,
    OllamaAPIClient,
)
import time
from amendements_intelligents.summary.summary_generation_load_balancer import (
    SummaryGenerationLoadBalancer,
)
from amendements_intelligents.summary.summary_prompt_builder import SummaryPromptBuilder

# Perf test for 50 concurrent calls to the LLM API
# 1 Albert client : 53.9 seconds
# 2 Albert clients : 37.7 seconds
# 4 Albert clients : 28.4 seconds
# 6 Albert clients : 24.6 seconds
# 8 Albert clients : 23.6 seconds
# 2 Ollama clients : 41.2 seconds
# 6 Ollama clients : 17.1 seconds
# 10 Ollama clients : 10.9 seconds
# 14 Ollama clients : 10.6 seconds

llm_api_clients = []

# Example setup for multiple clients
OLLAMA_USER = os.getenv("OLLAMA_USER")
OLLAMA_PASSWORD = os.getenv("OLLAMA_PASSWORD")
OLLAMA_ENDPOINT = "https://91.134.11.19.nip.io/api/generate"
# OLLAMA_ENDPOINT = os.getenv("OLLAMA_ENDPOINT")
OLLAMA_MODEL_NAME = os.getenv("OLLAMA_MODEL_NAME")
# OLLAMA_MODEL_NAME = "llama3.2"

for i in range(14):
    ollama_api_client = OllamaAPIClient(
        endpoint="https://91.134.11.19.nip.io/api/generate",
        model_name=OLLAMA_MODEL_NAME,
        user=OLLAMA_USER,
        password=OLLAMA_PASSWORD,
    )
    llm_api_clients.append(ollama_api_client)

# for i in range(6):
#     albert_api_client = EtalabAPIClient(
#         base_url=os.getenv("ETALAB_BASE_URL", "https://albert.api.etalab.gouv.fr/v1"),
#         api_key=os.getenv("ETALAB_API_KEY"),
#         model_name=os.getenv(
#             "ETALAB_MODEL_NAME", "meta-llama/Meta-Llama-3.1-70B-Instruct"
#         ),
#     )
#     llm_api_clients.append(albert_api_client)

# Instantiate the load balancer with the clients
load_balancer = SummaryGenerationLoadBalancer(clients=llm_api_clients)


prompts = [None] * 50
for i in range(50):
    prompt = SummaryPromptBuilder.build_prompt_new(
        explanatory_statement="Cet amendement vise à renforcer les mesures de prévention en matière de santé publique. Il propose d'augmenter le budget alloué aux campagnes de sensibilisation et de vaccination, ainsi que de financer des programmes de recherche sur les maladies infectieuses. En outre, il prévoit la création de centres de santé communautaires dans les zones rurales pour améliorer l'accès aux soins. Cet amendement est essentiel pour garantir une meilleure protection de la santé de nos concitoyens et pour prévenir les épidémies futures.",
        amdt_body="Ajouter un alinéa ainsi rédigé : « Art. L. 1311-1. – Le Gouvernement élabore un plan national de prévention en matière de santé publique. Ce plan définit les objectifs et les moyens de la politique de prévention en matière de santé publique. Il est mis en œuvre par les agences régionales de santé et les collectivités territoriales. »",
    )
    prompts[i] = prompt

start_time = time.time()

# Generate summaries concurrently
# prompts = [f"Give a one word answer. What comes after {i}?" for i in range(100)]
results = load_balancer.generate_summaries_concurrent(prompts)

end_time = time.time()
elapsed_time = end_time - start_time

# Print results
for i, result in enumerate(results):
    print(f"Response {i}: {result}")

print(f"Time taken for 50 calls: {elapsed_time} seconds")


Response 0: Élaborer un plan national de prévention en matière de santé publique mis en œuvre par les ARS et les collectivités territoriales.
Response 1: Mettre en œuvre un plan national de prévention en matière de santé publique.
Response 2: Élaborer un plan national de prévention en matière de santé publique mis en œuvre par les ARS et les collectivités territoriales.
Response 3: Mettre en œuvre un plan national de prévention en matière de santé publique.
Response 4: Mettre en œuvre un plan national de prévention en matière de santé publique.
Response 5: Élaborer un plan national de prévention en matière de santé publique mis en œuvre par les ARS et les collectivités territoriales.
Response 6: Mettre en œuvre un plan national de prévention en matière de santé publique par les ARS et les collectivités territoriales.
Response 7: Mettre en œuvre un plan national de prévention en matière de santé publique par les ARS et les collectivités territoriales.
Response 8: Mettre en œuvre un plan